In [1]:
!pip install vllm==0.8.5 langchain==0.3.19 langchain_community==0.3.18 chromadb==1.0.7 langchain-huggingface==0.1.2 -qq

In [2]:
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import Chroma
from langchain.chains import RetrievalQA
from langchain_community.llms import VLLM
from langchain_community.document_loaders import WebBaseLoader

# 1. load document
loader = WebBaseLoader(web_paths=("https://aws.amazon.com/cn/what-is/retrieval-augmented-generation/",), )
documents = loader.load()

# 2. split document
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0)
texts = text_splitter.split_documents(documents)

# 3. choose Embedding model
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# 4. choose Chroma as vector database and convert texts to embedding, then add embedding into the vector database
db = Chroma.from_documents(texts, embeddings, persist_directory="doc_db")
db.persist()

# 5. create llm
model = "Qwen/Qwen2.5-1.5B-Instruct"
llm = VLLM(model=model,
           trust_remote_code=True,
           max_new_tokens=520,
           top_k=10,
           top_p=0.95,
           temperature=0.8,
           dtype='half'
           # tensor_parallel_size=... # for distributed inference
           )

# 6. simple way to use rag
# code ref https://python.langchain.com/v0.2/docs/tutorials/rag/
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.prompts import ChatPromptTemplate

template = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.
Question: {question}
Context: {context}
Answer:
"""
prompt = ChatPromptTemplate.from_template(template)

retriever = db.as_retriever()


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
)

response = rag_chain.invoke("什么是rag?")
print("response:\n", response)


<ipython-input-2-ac77539c71ca>:17: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.wa

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

INFO 05-02 09:00:29 [loader.py:458] Loading weights took 3.57 seconds
INFO 05-02 09:00:30 [model_runner.py:1140] Model loading took 2.8881 GiB and 21.069554 seconds
INFO 05-02 09:00:35 [worker.py:287] Memory profiling takes 4.87 seconds
INFO 05-02 09:00:35 [worker.py:287] the current vLLM instance can use total_gpu_memory (14.74GiB) x gpu_memory_utilization (0.90) = 13.27GiB
INFO 05-02 09:00:35 [worker.py:287] model weights take 2.89GiB; non_torch_memory takes 0.02GiB; PyTorch activation peak memory takes 2.02GiB; the rest of the memory reserved for KV Cache is 8.34GiB.
INFO 05-02 09:00:36 [executor_base.py:112] # cuda blocks: 19523, # CPU blocks: 9362
INFO 05-02 09:00:36 [executor_base.py:117] Maximum concurrency for 32768 tokens per request: 9.53x
INFO 05-02 09:00:41 [model_runner.py:1450] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI.

Capturing CUDA graph shapes:   0%|          | 0/35 [00:00<?, ?it/s]

INFO 05-02 09:01:23 [model_runner.py:1592] Graph capturing finished in 42 secs, took 0.19 GiB
INFO 05-02 09:01:23 [llm_engine.py:437] init engine (profile, create kv cache, warmup model) took 52.75 seconds


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

response:
 RAG是一种将大型语言模型(LLM)与权威的知识来源(如数据库、文献或社交媒体提要)相结合的方法。它通过重定向LLM，从知识源中检索相关信息来增强LLM的生成能力。RAG可以提高LLM的经济效率、提供最新信息、增强用户信任度和增加开发人员的控制权。它通过使用语义搜索技术来处理大型数据集，提高输出的质量。此外，RAG允许LLM更新其知识库以反映实时信息，从而维护准确性和相关性。AWS提供了多种解决方案，如Amazon Bedrock和Amazon Kendra，支持RAG的需求，简化了RAG的实施和管理。RAG和语义搜索的主要区别在于，语义搜索通过自动处理数据，提高了检索结果的质量和相关性，而RAG则通过增强提示工程技术，使大型语言模型能够更好地理解和生成用户查询的相关信息。


In [3]:
# 7. advanced way to use rag:create RetrievalQA chain
qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db.as_retriever(search_kwargs={"k": 3}),
    return_source_documents=True,
)

# 8. ask a question
query = "什么是rag？"
result = qa({"query": query})

print("Question:", query)
print("Answer:", result["result"])

# 9. option: print source document
if "source_documents" in result:
    print("\nSource Documents:")
    for doc in result["source_documents"]:
        print(doc.page_content)
        print("-" * 20)

<ipython-input-3-7b2a3ca44ad2>:11: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result = qa({"query": query})


Processed prompts:   0%|          | 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Question: 什么是rag？
Answer:  RAG是解决其中一些挑战的一种方法。它会重定向LLM，从权威的、预先确定的知识来源中检索相关信息。组织可以更好地控制生成的文本输出，并且用户可以深入了解LLM 如何生成响应。  例如，Amazon Kendra 是一种由机器学习支持的高精度企业搜索服务，它提供了经过优化的Kendra检索API，您可以将其与Amazon Kendra的高精度语义排名器一起使用，作为RAG工作流程的企业检索器。例如，使用检索API，您可以检索多达100个语义相关的段落，每个段落最多包含200个标记词，按相关性排序。 使用检索API，您可以检索多达100个语义相关的段落，每个段落最多包含200个标记词，按相关性排序。  使用预构建的连接器连接到常用数据技术，例如Amazon Simple Storage Service、SharePoint、Confluence和其他网站。支持多种文档格式，例如HTML、Word、PowerPoint、PDF、Excel和文本文件。根据最终用户权限允许的文档筛选响应。  组织可以更自信地为更广泛的应用程序实施生成式人工智能技术。  组织可以更好地控制生成的文本输出，并且用户可以深入了解LLM 如何生成响应。  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API  语义搜索API  搜索API  生成API

Source Documents:
您可以将大型语言模型看作是一个过于热情的新员工，他拒绝随时了解